# Experiment Secondary Aerosols: Volcanic eruption with internally mixed aerosols

### Objective:
- Learn how to run a simulation with internally mixed aerosols how to consider nucleation, condensation, and coagulation

### Simulation Design:
- Run one global simulation for the 2019 Raikoke eruption

### Analysis:
- Plot and understand the formation of sulfate and the aging of ash aerosols

### Necessary xml files:

* pntSrc.xml     : for point source emission
* chemtracer.xml : for simplified OH-chemistry
* modes.xml      : for describtion of the (background) modes
* aerotracer.xml : for describtion of the treated aerosol tracers
* coagulate.xml  : for the coagulation matrix

## 1. Prepare the emission
We want to simulate the Raikoke eruption in 2019 and we assume the following eruption source parameters:
* Ash source strength of 59000 kg/s (distributed equally over three modes) from 21 June 2019, 18-03 UTC
* Plume height and bottom height of 14000 m and 8000 m, respectively (emission profile will then be constant between bottom and top of the plume)
* Location of the eruption: 153.24°E and 49.29°N 
* SO$_2$ source strength of 46300 kg/s over the same period and with the same emission profile

The emitted ash distribution consists of 3 modes with the following median diameters and standard deviations:

| | Insoluble Accumulation | Insoluble Coarse | Insoluble Giant |
|-----|-----|-----|-----|
| median diameter [m] (dg3\_emiss)| 0.8E-6 | 2.98E-6 | 11.35E-6 |
| standard deviation (sigma\_emiss)| 1.4 | 1.4 | 1.4 |
| emission strength [kg/s] (source\_strength)| 19700 | 19700 | 19700 |




## 2. Prepare the internally mixed aerosols and activate processes


1. In Ex02_modes.xml: 
* activate the processes condensation and coagulation for all modes except the giant mode
* enable shifting to the mixed mode for the insoluble accumlation and insoluble coarse mode (replace the ??? by the mode it should be shifted to, e.g. for insol_acc insert mixed_acc)
* enable shifting to the larger mode for the soluble Aitken mode (replace the ???)
2. In the Ex02_aerotracer.xml: 
* activate nucleation for SO4 in the Aitken mode 
* deactivate nucleation for SO4 in the other modes (--> nucleation of sulfate only into the smallest mode)
3. Understand the coagulation matrix in Ex02_coagulate.xml


## 3. Input data
Insert the correct paths to link the input data

In [ ]:
# Output directory
export OUTDIR=/path/to/scratch/

# Path to the xml files
export XMLDIR=/path/to/xmlfiles/

# input data
export DATADIR=/path/to/input-data

# ICON-ART code directory
export ICONDIR=path/to/icon-model
export ARTDIR=${ICONDIR}/externals/art

In [ ]:
if [ ! -d $OUTDIR ]; then
    mkdir -p $OUTDIR
fi

cd $OUTDIR

# copy icon binary to OUTDIR
cp ${ICONDIR}/bin/icon icon.exe

In [ ]:
# ICON Initial and Grid data
ln -sf ${DATADIR}/icon_grid_0030_R02B05_G.nc ${OUTDIR}/iconR2B05-DOM01.nc
ln -sf ${DATADIR}/icon_grid_0030_R02B05_G-grfinfo.nc ${OUTDIR}/iconR2B05-DOM01-grfinfo.nc
ln -sf ${DATADIR}/icon_extpar_0030_R02B05_G_tiles.nc ${OUTDIR}/extpar_iconR2B05-DOM01.nc
ln -sf ${DATADIR}/icon_init_0030_R02B05_2019062112.nc ${OUTDIR}/igfff00000000.nc

ln -sf ${ICONDIR}/data/rrtmg_lw.nc ${OUTDIR}/rrtmg_lw.nc
ln -sf ${ICONDIR}/data/ECHAM6_CldOptProps.nc ${OUTDIR}/ECHAM6_CldOptProps.nc

# Dictionary for the mapping: DWD GRIB2 names <-> ICON internal names
ln -sf ${ICONDIR}/run/ana_varnames_map_file.txt ${OUTDIR}/map_file.ana

# aerosols
ln -sf ${ARTDIR}/runctrl_examples/init_ctrl/PrescribedOpticalProperties_CMD_USE.nc ${OUTDIR}/prescribed_opt_props.nc

## chemistry
ln -sf ${ARTDIR}/runctrl_examples/init_ctrl/Simnoy2002.dat ${OUTDIR}/Simnoy2002.dat
ln -sf ${ARTDIR}/runctrl_examples/init_ctrl/Linoz2004Br.dat ${OUTDIR}/Linoz2004Br.dat

ln -sf ${ARTDIR}/runctrl_examples/photo_ctrl/FJX_scat-aer.dat      ${OUTDIR}/FJX_scat-aer.dat
ln -sf ${ARTDIR}/runctrl_examples/photo_ctrl/FJX_j2j.dat           ${OUTDIR}/FJX_j2j.dat
ln -sf ${ARTDIR}/runctrl_examples/photo_ctrl/FJX_j2j_extended.dat  ${OUTDIR}/FJX_j2j_extended.dat
ln -sf ${ARTDIR}/runctrl_examples/photo_ctrl/FJX_scat-cld.dat      ${OUTDIR}/FJX_scat-cld.dat
ln -sf ${ARTDIR}/runctrl_examples/photo_ctrl/FJX_scat-ssa.dat      ${OUTDIR}/FJX_scat-ssa.dat
ln -sf ${ARTDIR}/runctrl_examples/photo_ctrl/FJX_scat-UMa.dat      ${OUTDIR}/FJX_scat-UMa.dat
ln -sf ${ARTDIR}/runctrl_examples/photo_ctrl/FJX_spec_extended.dat ${OUTDIR}/FJX_spec_extended.dat
ln -sf ${ARTDIR}/runctrl_examples/photo_ctrl/FJX_spec_extended_lyman.dat ${OUTDIR}/FJX_spec_extended_lyman.dat
ln -sf ${ARTDIR}/runctrl_examples/photo_ctrl/FJX_spec.dat          ${OUTDIR}/FJX_spec.dat
ln -sf ${ARTDIR}/runctrl_examples/photo_ctrl/atmos_std.dat         ${OUTDIR}/atmos_std.dat
ln -sf ${ARTDIR}/runctrl_examples/photo_ctrl/atmos_h2och4.dat      ${OUTDIR}/atmos_h2och4.dat

## 4. The runscript


In [ ]:
cat > icon_master.namelist << EOF
&master_nml
 lRestart               = .false.
/
&master_time_control_nml
 experimentStartDate = "2019-06-21T12:00:00"
 forecastLeadTime = "P2D"
 checkpointTimeIntval = "P10D"
/
&master_model_nml
  model_type=1
  model_name="ATMO"
  model_namelist_filename="NAMELIST_Raikoke-June-2019"
  model_min_rank=1
  model_max_rank=65536
  model_inc_rank=1
/
&time_nml
 ini_datetime_string = "2019-06-21T12:00:00"
 dt_restart          = 864000   ! 10 days
/
EOF

In [ ]:
cat > NAMELIST_Raikoke-June-2019 << EOF
&parallel_nml
 nproma         = 8  ! optimal setting 8 for CRAY; use 16 or 24 for IBM
 p_test_run     = .false.
 l_test_openmp  = .false.
 l_log_checks   = .false.
 num_io_procs   = 1   ! up to one PE per output stream is possible
 iorder_sendrecv = 3  ! best value for CRAY (slightly faster than option 1)
/
&grid_nml
 dynamics_grid_filename  = 'iconR2B05-DOM01.nc'
 dynamics_parent_grid_id = 0
 lredgrid_phys           = .false.
/
&initicon_nml
 init_mode                   =   7                           ! 2: IFS; 7: Initialized DWD Analysis
 lread_ana                   =       .FALSE.                 ! no analysis data will be read
 dwdfg_filename              =       "igfff00000000.nc"         ! initial data filename
 filetype                    =   4
 ana_varnames_map_file       =       "map_file.ana"          ! dictionary mapping internal names onto GRIB2 shortNames
 ltile_coldstart             =       .TRUE.                  ! coldstart for surface tiles
 ltile_init                  =       .FALSE.                 ! set it to .TRUE. if FG data originate from run without tiles
/
&run_nml
 num_lev        = 90
 lvert_nest     = .true.       ! use vertical nesting if a nest is active
 !nsteps         = ${nsteps}   ! 50 ! 1200 ! 7200 !
 dtime          = 360          ! timestep in seconds
 ldynamics      = .TRUE.       ! dynamics
 ltransport     = .true.
 iforcing       = 3            ! NWP forcing
 ltestcase      = .false.      ! false: run with real data
 msg_level      = 7            ! print maximum wind speeds every 5 time steps
 ltimer         = .true.       ! set .TRUE. for timer output
 timers_level   = 15           ! can be increased up to 10 for detailed timer output
 output         = "nml"
 lart           = .true.
/
&nwp_phy_nml
 inwp_gscp       = 1
 inwp_convection = 1
 inwp_radiation  = 4
 inwp_cldcover   = 1
 inwp_turb       = 1
 inwp_satad      = 1
 inwp_sso        = 1
 inwp_gwd        = 1
 inwp_surface    = 1
 icapdcycl       = 3 ! apply CAPE modification to improve diurnalcycle over tropical land (optimizes NWP scores)
 latm_above_top  = .false.  ! the second entry refers to the nested domain (if present)
 efdt_min_raylfric = 7200.
 itype_z0         = 2
 icapdcycl        = 3
 icpl_aero_conv   = 1        ! check meaning -> default 0 - off
 icpl_aero_gscp   = 1        ! check meaning -> default 0 - off
 icpl_o3_tp       = 1
 ! resolution-dependent settings - please choose the appropriate one
 dt_rad    = 1440.
 dt_conv   = 360.,360.,180.
 dt_sso    = 720.
 dt_gwd    = 720.
/
&nwp_tuning_nml
 itune_albedo  = 1
 tune_gkwake   = 1.5
 tune_gfrcrit  = 0.425
 tune_grcrit   = 0.5
 tune_gkdrag   = 0.16
 tune_zvz0i    = 0.85
 tune_box_liq_asy = 3.25
 tune_minsnowfrac = 0.2
 tune_gfluxlaun  = 3.75e-3
 tune_rcucov = 0.075
 tune_rhebc_land = 0.825
/
&turbdiff_nml
 tkhmin       = 0.6
 tkhmin_strat = 1.0
 tkmmin       = 0.75
 tkmmin_strat = 1.5
 pat_len      =  750.
 c_diff       =  0.2
 rat_sea      =  0.8
 rlam_heat    = 10.
 ltkesso      = .TRUE.
 frcsmot      = 0.2
 imode_frcsmot = 2
 icldm_turb   = 1
 itype_sher   = 1
 ltkeshs      = .TRUE.
 a_hshr       = 2.0
/
&lnd_nml
  ntiles         = 3
  nlev_snow      = 1
  lmulti_snow    = .FALSE.
  itype_heatcond = 3
  idiag_snowfrac = 20
  itype_snowevap = 2
  itype_canopy   = 2
  lsnowtile      = .TRUE.
  lseaice        = .TRUE.
  llake          = .TRUE.
  itype_lndtbl   = 4
  itype_evsl     = 4
  itype_root     = 2
  cwimax_ml      = 5.e-4
  c_soil         = 1.25
  c_soil_urb     = 0.5
  sstice_mode    = 2
  lprog_albsi    = .TRUE.
/
&radiation_nml
 irad_o3       = 7
 irad_aero     = 6
 albedo_type   = 2 ! Modis albedo
 vmr_co2       = 390.e-06 ! values representative for 2012
 vmr_ch4       = 1800.e-09
 vmr_n2o       = 322.0e-09
 vmr_o2        = 0.20946
 vmr_cfc11     = 240.e-12
 vmr_cfc12     = 532.e-12
 ecrad_data_path = "${ICONDIR}/externals/ecrad/data"
/
&nonhydrostatic_nml
 itime_scheme   = 4
 vwind_offctr   = 0.2
 damp_height    = 50000.
 rayleigh_coeff = 0.1
 divdamp_order  = 24    ! for data assimilation runs, '2' provides extra-strong filtering of gravity waves
 divdamp_type   = 32    !!! optional: 2 for assimilation cycle if very strong gravity-wave filtering is desired
 divdamp_fac    = 0.004
 igradp_method  = 3
 l_zdiffu_t     = .true.
 thslp_zdiffu   = 0.02
 thhgtd_zdiffu  = 125.
 htop_moist_proc= 22500.
 hbot_qvsubstep = 19000. ! use 19000. with R3B7
 htop_aero_proc  = 22500.
/
&sleve_nml
 min_lay_thckn   = 20.
 max_lay_thckn   = 400.   ! maximum layer thickness below htop_thcknlimit
 htop_thcknlimit = 14000. ! this implies that the upcoming COSMO-EU nest will have 60 levels
 top_height      = 75000.
 stretch_fac     = 0.9
 decay_scale_1   = 4000.
 decay_scale_2   = 2500.
 decay_exp       = 1.2
 flat_height     = 16000.
/
&dynamics_nml
 divavg_cntrwgt = 0.50
 lcoriolis      = .TRUE.
/
&transport_nml
 ivadv_tracer  = 3,3,3,3,3,3,3,3,3,3,3
 itype_hlimit  = 3,4,4,4,4,3,3,3,3,3,3
 ihadv_tracer  = 52,2,2,2,2,22,22,22,22,22,22
 iadv_tke      = 0
/
&diffusion_nml
 hdiff_order      = 5
 itype_vn_diffu   = 1
 itype_t_diffu    = 2
 hdiff_efdt_ratio = 24.0
 hdiff_smag_fac   = 0.025
 lhdiff_vn        = .TRUE.
 lhdiff_temp      = .TRUE.
/
&interpol_nml
 nudge_zone_width  = 8
 lsq_high_ord      = 3
 l_intp_c2l        = .true.
 l_mono_c2l        = .true.
/
&extpar_nml
 itopo          = 1
 n_iter_smooth_topo = 1
 heightdiff_threshold = 3000.
/
&io_nml
 itype_pres_msl = 5  ! 5: new DWD method; 4: IFS method with bug fix for self-consistency between SLP and geopotential
 itype_rh       = 1  ! RH w.r.t. water
 restart_write_mode = 'joint procs multifile'
/
&output_nml
 filetype                     = 4              ! output format: 2=GRIB2, 4=NETCDFv2
 dom                          = -1                       ! write all domains
 mode                         = 1                        ! 1 = forecast mode with TAXIS_RELATIVE, works only with output in multiples of 1h; 2 = climate mode, default, TAXIS_ABSOLUTE
 output_bounds                = 0., 691200., 3600.       ! start, end, increment
 steps_per_file               = 1
 include_last                 = .TRUE.
 output_filename              = 'Raikoke-June-2019-forecast_mode-remap'                ! file name base
 ml_varlist                   = 'TRSO2_chemtr','TRH2SO4_chemtr','OH_Nconc','group:ART_AEROSOL','rho','pres','temp', 'z_mc','z_ifc'
 remap                        = 1
 reg_lon_def                  = 120.,0.25,280.
 reg_lat_def                  = 85.,-0.25, 30.
/
&art_nml
 lart_diag_out   = .TRUE.
 lart_aerosol    = .TRUE.
 lart_pntSrc     = .TRUE.
 lart_chem       = .TRUE.
 lart_chemtracer = .TRUE.
 iart_init_gas  = 1
 iart_ari            = 0
 iart_modeshift      = 1          ! 0 = off; 1 = on
! iart_isorropia      = 1          ! 0 = off; 1 = on
 iart_aero_washout   = 1

 cart_aerosol_xml    = '${XMLDIR}/aerotracer.xml'
 cart_modes_xml      = '${XMLDIR}/modes.xml'
 cart_coag_xml       = '${XMLDIR}/coagulate.xml'
 cart_pntSrc_xml     = '${XMLDIR}/pntSrc.xml'
 cart_chemtracer_xml = '${XMLDIR}/chemtracer.xml'
 cart_opt_props_nc   = '${OUTDIR}/prescribed_opt_props.nc'
/
EOF


## 5. Prepare the ICON-ART batch job on Levante
Change to your project account

In [ ]:
cat > ${OUTDIR}/job_ICON << ENDFILE
#!/bin/bash
#SBATCH --job-name=EXP01_ART          # Specify job name
#SBATCH --partition=compute            # Specify partition name
#SBATCH --nodes=4                    # Specify number of nodes
#SBATCH --ntasks-per-node=128          # Specify number of (MPI) tasks on each node
#SBATCH --time=01:00:00                 # Set a limit on the total run time
#SBATCH --account=????                # Charge resources on this project account

unset SLURM_EXPORT_ENV 
unset SLURM_MEM_PER_NODE
unset SBATCH_EXPORT

export LD_LIBRARY_PATH="/usr/lib:/usr/lib64:/sw/spack-levante/netcdf-c-4.8.1-2k3cmu/lib:/sw/spack-levante/netcdf-fortran-4.5.3-k6xq5g/lib:/sw/spack-levante/hdf5-1.12.1-tvymb5/lib:/sw/spack-levante/eccodes-2.21.0-3ehkbb/lib64:/sw/spack-levante/intel-oneapi-mkl-2022.0.1-ttdktf/mkl/2022.0.1/lib/intel64/"

export OMPI_MCA_pml="ucx"
export OMPI_MCA_btl=self
export OMPI_MCA_osc="pt2pt"
export UCX_IB_ADDR_TYPE=ib_global

export OMPI_MCA_coll="^ml,hcoll"
export OMPI_MCA_coll_hcoll_enable="0"
export HCOLL_ENABLE_MCAST_ALL="0"
export HCOLL_MAIN_IB=mlx5_0:1
export UCX_NET_DEVICES=mlx5_0:1
export UCX_TLS=mm,knem,cma,dc_mlx5,dc_x,self
export UCX_UNIFIED_MODE=y
export HDF5_USE_FILE_LOCKING=FALSE
export OMPI_MCA_io="romio321"
export UCX_HANDLE_ERRORS=bt


ulimit -s 102400
ulimit -c 0

srun -l --cpu_bind=cores --distribution=block:cyclic --propagate=STACK,CORE ./icon.exe


ENDFILE


## 7. Submit the job

In [ ]:
cd $OUTDIR && chmod +x job_ICON && sbatch job_ICON

## 8. Check the job status

In [ ]:
squeue -u $USER
squeue -u $USER --start

## 9. Plotting

Open [Plot_Secondary_Aerosols.ipynb](Plot_Secondary_Aerosols.ipynb) and plot the data.